# Deploy Multi-Agent Financial Advisor on Amazon Bedrock AgentCore Runtime

This notebook deploys the multi-agent financial advisory system (from `finance_multi_agent.ipynb` in Lab2) to **Amazon Bedrock AgentCore Runtime** — a serverless, purpose-built hosting environment for AI agents.

## What This Lab Does

- Deploys the **orchestrator agent** (Claude Haiku 4.5 via Bedrock) and **financial analysis agent** (Qwen 3.5 9B via SageMaker OpenAI-compatible API) as a containerized service on AgentCore Runtime
- Uses **IAM SigV4** authentication — invoke via `boto3` with standard AWS credentials
- Includes full observability (CloudWatch Logs, X-Ray traces, token usage metrics)

## Authentication: IAM SigV4

AgentCore Runtime uses IAM SigV4 by default:
- Invoke with `boto3` client directly (`bedrock-agentcore` service)
- Standard IAM permissions control access
- No additional auth setup required

## SageMaker OpenAI Endpoint Inside AgentCore

When the agent container runs in AgentCore Runtime:
- It gets an **execution role** with the permissions you configure
- The role needs `sagemaker:InvokeEndpoint` permission on the Qwen endpoint
- `sagemaker.core.token_generator.generate_token()` works because it reads from the container's role credentials
- The `httpx.AsyncClient` + `SageMakerAuth` pattern auto-refreshes tokens from the role

## Prerequisites

- Qwen 3.5 9B endpoint deployed on SageMaker (from the deployment notebook)
- IAM role with AgentCore and SageMaker permissions
- Amazon Bedrock model access for Claude Haiku 4.5

In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts bedrock-agentcore-starter-toolkit boto3 strands-agents python-dotenv

In [ ]:
import uuid
import json
import os
import boto3
from pathlib import Path
from boto3.session import Session
from dotenv import load_dotenv
from bedrock_agentcore_starter_toolkit import Runtime

# Load endpoint config from .env file
load_dotenv(Path("..") / ".env", override=True)

boto_session = Session()
region = os.environ["AWS_REGION"]

print(f"Region: {region}")

## Step 1: Prepare Agent Code for AgentCore Runtime

AgentCore requires:
- A `main.py` with `BedrockAgentCoreApp` + `@app.entrypoint` decorator
- A `requirements.txt` for container dependencies
- A `budget_agent.py` module (copied from Lab2)

The agent inside the container uses the **OpenAI-compatible SageMaker endpoint** pattern with auto-refreshing tokens.

The `budget_agent.py` module is defined in `Lab2/strands_agents/` and used by `main.py` at runtime. Since AgentCore packages only the files in the current directory into the container, we need to copy it here before building.

In [ ]:
# Copy budget_agent.py from Lab2/strands_agents into this folder for container packaging
import shutil
from pathlib import Path

src = Path("..") / "Lab2" / "strands_agents" / "budget_agent.py"
dst = Path(".") / "budget_agent.py"

shutil.copy2(src, dst)
print(f"✓ Copied {src} → {dst}")

In [ ]:
%%writefile main.py
# AgentCore Runtime deployment — Multi-Agent Financial Advisor
# Financial Analysis Agent uses Qwen 3.5 9B via SageMaker OpenAI-compatible API

import os

# Enable OpenTelemetry instrumentation for Strands Agents
os.environ.setdefault("AGENT_OBSERVABILITY_ENABLED", "true")

import yfinance as yf
import httpx
from openai import AsyncOpenAI
from typing import List

from strands import Agent, tool
from strands.models import BedrockModel
from strands.models.openai import OpenAIModel
from strands.agent.conversation_manager import SummarizingConversationManager
from sagemaker.core.token_generator import generate_token
from bedrock_agentcore import BedrockAgentCoreApp

from budget_agent import FinancialReport

# --- Configuration ---
from pathlib import Path
from dotenv import load_dotenv

# Load endpoint config from .env file (shared with deployment notebook)
load_dotenv(Path(__file__).resolve().parent.parent / ".env", override=True)

SAGEMAKER_ENDPOINT_NAME = os.environ["SAGEMAKER_ENDPOINT_NAME"]
SAGEMAKER_REGION = os.environ["AWS_REGION"]


# --- SageMaker OpenAI-Compatible Client Setup ---
class SageMakerAuth(httpx.Auth):
    """Auto-refreshing auth that generates a fresh bearer token on each request."""
    def __init__(self, region: str):
        self.region = region

    def auth_flow(self, request):
        request.headers["Authorization"] = f"Bearer {generate_token(region=self.region)}"
        yield request


base_url = f"https://runtime.sagemaker.{SAGEMAKER_REGION}.amazonaws.com/endpoints/{SAGEMAKER_ENDPOINT_NAME}/openai/v1"
async_http_client = httpx.AsyncClient(auth=SageMakerAuth(region=SAGEMAKER_REGION))
strands_client = AsyncOpenAI(
    base_url=base_url,
    api_key="sagemaker",
    http_client=async_http_client,
)

qwen_model = OpenAIModel(
    client=strands_client,
    model_id="",
    params={
        "temperature": 0.7,
        "max_tokens": 4096,
        "stream_options": {"include_usage": True},
    },
)


# --- Financial Analysis Tools ---
@tool
def get_stock_analysis(symbol: str) -> str:
    """Get comprehensive analysis for a specific stock symbol."""
    try:
        stock = yf.Ticker(symbol)
        info = stock.info
        hist = stock.history(period="1y")
        if hist.empty:
            return f"No data available for {symbol}"
        current_price = hist["Close"].iloc[-1]
        year_high = hist["High"].max()
        year_low = hist["Low"].min()
        avg_volume = hist["Volume"].mean()
        price_change = ((current_price - hist["Close"].iloc[0]) / hist["Close"].iloc[0]) * 100
        return (
            f"Stock Analysis for {symbol.upper()}: "
            f"Price=${current_price:.2f}, 52W High=${year_high:.2f}, 52W Low=${year_low:.2f}, "
            f"YTD Change={price_change:.2f}%, Avg Volume={avg_volume:,.0f}, "
            f"Company={info.get('longName', 'N/A')}, Sector={info.get('sector', 'N/A')}"
        )
    except Exception as e:
        return f"Unable to retrieve data for {symbol}: {str(e)}"


@tool
def create_diversified_portfolio(risk_level: str, investment_amount: float) -> str:
    """Create a diversified portfolio based on risk level (conservative, moderate, aggressive) and investment amount."""
    portfolios = {
        "conservative": {"stocks": ["AAPL", "MSFT", "JNJ", "PG", "KO"], "weights": [0.25, 0.25, 0.20, 0.15, 0.15]},
        "moderate": {"stocks": ["AAPL", "GOOGL", "AMZN", "TSLA", "NVDA"], "weights": [0.30, 0.25, 0.20, 0.15, 0.10]},
        "aggressive": {"stocks": ["TSLA", "NVDA", "AMZN", "GOOGL", "META"], "weights": [0.30, 0.25, 0.20, 0.15, 0.10]},
    }
    if risk_level.lower() not in portfolios:
        return "Risk level must be: conservative, moderate, or aggressive"
    p = portfolios[risk_level.lower()]
    result = f"{risk_level.upper()} Portfolio (${investment_amount:,.0f}): "
    for stock, weight in zip(p["stocks"], p["weights"]):
        result += f"{stock}={weight*100:.0f}%(${investment_amount*weight:,.0f}) "
    return result


@tool
def compare_stock_performance(symbols: List[str], period: str = "1y") -> str:
    """Compare performance of multiple stocks over a specified period."""
    if len(symbols) > 5:
        return "Please limit comparison to 5 stocks maximum"
    try:
        results = []
        for symbol in symbols:
            hist = yf.Ticker(symbol).history(period=period)
            if not hist.empty:
                perf = ((hist["Close"].iloc[-1] - hist["Close"].iloc[0]) / hist["Close"].iloc[0]) * 100
                results.append(f"{symbol}: {perf:+.2f}%")
        return f"Performance Comparison ({period}): " + ", ".join(results)
    except Exception as e:
        return f"Error comparing stocks: {str(e)}"


# --- Financial Analysis Agent Prompt ---
FINANCIAL_ANALYSIS_PROMPT = """You are a specialized financial analysis agent focused on investment research and portfolio recommendations.
Research and analyze stock data, create diversified portfolios, and provide data-driven recommendations.
Always include disclaimers about market risks."""


# --- Agent Tools for Orchestrator ---
@tool
def budget_agent_tool(query: str) -> FinancialReport:
    """Generate structured financial reports with budget analysis and recommendations."""
    try:
        # Import tools from budget_agent module and create a fresh instance
        from budget_agent import (
            calculate_budget_breakdown, analyze_spending_pattern, calculator,
            BUDGET_AGENT_PROMPT,
        )
        fresh_budget_agent = Agent(
            model=BedrockModel(
                model_id="global.anthropic.claude-sonnet-4-6-20250514-v1:0",
                region_name=SAGEMAKER_REGION,
                temperature=0.0,
            ),
            system_prompt=BUDGET_AGENT_PROMPT,
            tools=[calculate_budget_breakdown, analyze_spending_pattern, calculator],
            callback_handler=None,
        )
        return fresh_budget_agent(query, output_model=FinancialReport)
    except Exception as e:
        return FinancialReport(
            monthly_income=0.0, budget_categories=[],
            recommendations=[f"Error: {str(e)}"], financial_health_score=1,
        )


@tool
def financial_analysis_agent_tool(query: str) -> str:
    """Handle investment analysis queries including stock research, portfolio creation, and performance comparisons."""
    from opentelemetry import trace

    tracer = trace.get_tracer("financial_analysis_agent")

    try:
        # Create a fresh agent instance per invocation to avoid
        # "Agent is already processing a request" concurrent invocation error
        fa_agent = Agent(
            model=OpenAIModel(
                client=strands_client,
                model_id="",
                params={
                    "temperature": 0.7,
                    "max_tokens": 4096,
                    "stream_options": {"include_usage": True},
                },
            ),
            system_prompt=FINANCIAL_ANALYSIS_PROMPT,
            tools=[get_stock_analysis, create_diversified_portfolio, compare_stock_performance],
            callback_handler=None,
        )

        with tracer.start_as_current_span("gen_ai.chat", attributes={
            "gen_ai.system": "openai",
            "gen_ai.request.model": f"qwen3.5-9b ({SAGEMAKER_ENDPOINT_NAME})",
            "gen_ai.operation.name": "chat",
        }) as span:
            result = fa_agent(query)
            # Extract token usage from accumulated_usage in agent metrics
            usage = result.metrics.accumulated_usage
            input_tokens = usage.get('inputTokens', 0)
            output_tokens = usage.get('outputTokens', 0)
            total_tokens = usage.get('totalTokens', 0)

            span.set_attribute("gen_ai.usage.input_tokens", input_tokens)
            span.set_attribute("gen_ai.usage.output_tokens", output_tokens)
            span.set_attribute("gen_ai.usage.total_tokens", total_tokens)

            response = str(result)

        return f"[Routed to: Qwen 3.5 9B on SageMaker endpoint '{SAGEMAKER_ENDPOINT_NAME}' via OpenAI-compatible API]\n\n{response}"
    except Exception as e:
        return f"Financial analysis error: {str(e)}"


# --- Orchestrator Agent (Claude Haiku 4.5 via Bedrock) ---
ORCHESTRATOR_PROMPT = """You are a financial advisor orchestrator coordinating specialized agents.
Use budget_agent_tool for budgets, spending, savings. Use financial_analysis_agent_tool for stocks, portfolios, investments.
You can use both together. Synthesize coherent answers with actionable next steps."""

orchestrator_model = BedrockModel(
    model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    region_name=SAGEMAKER_REGION,
    temperature=0.0,
)


# --- AgentCore Runtime Entrypoint ---
app = BedrockAgentCoreApp()


@app.entrypoint
async def invoke(payload):
    """AgentCore Runtime invocation handler with streaming."""
    # Create a fresh orchestrator per request to avoid concurrent invocation errors
    orchestrator = Agent(
        model=orchestrator_model,
        system_prompt=ORCHESTRATOR_PROMPT,
        tools=[budget_agent_tool, financial_analysis_agent_tool],
        conversation_manager=SummarizingConversationManager(
            summary_ratio=0.3, preserve_recent_messages=5,
        ),
    )
    user_message = payload["prompt"]
    async for event in orchestrator.stream_async(user_message):
        if "data" in event:
            yield event["data"]


if __name__ == "__main__":
    app.run()



In [ ]:
%%writefile requirements.txt
strands-agents[otel]>=1.0.0
boto3
botocore
openai
httpx
yfinance
pydantic
bedrock-agentcore
sagemaker-core
python-dotenv

## Step 2: Test Locally

Before deploying to AgentCore, verify the agent server works locally:

**Terminal 1** — Start the server:
```bash
python main.py
```

**Terminal 2** — Send a test request:
```bash
curl -X POST http://localhost:8080/invocations \
   -H "Content-Type: application/json" \
   -d '{"prompt": "Analyze Apple stock"}'
```

## Step 3: Configure and Deploy to AgentCore Runtime

AgentCore uses IAM SigV4 authentication by default.

In [ ]:
# Configure AgentCore Runtime
agentcore_runtime = Runtime()
agent_name = "personal_finance_agent"

print("Configuring AgentCore Runtime (IAM SigV4 auth)...")
response = agentcore_runtime.configure(
    entrypoint="main.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
)

print("Configuration completed ✓")

### Important: Add SageMaker permissions to the execution role

The auto-created execution role needs permission to invoke the SageMaker endpoint and generate bearer tokens for the OpenAI-compatible API. The cell below automatically finds the role and attaches the required policy.

Permissions added:
- `sagemaker:InvokeEndpoint` — standard inference
- `sagemaker:InvokeEndpointWithResponseStream` — streaming inference
- `sagemaker:CallWithBearerToken` — required for OpenAI-compatible API path

In [ ]:
import json
import boto3

iam = boto3.client("iam")
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]

# Find the auto-created AgentCore execution role
# The toolkit names it: AmazonBedrockAgentCoreSDKRuntime-<region>-<hash>
paginator = iam.get_paginator("list_roles")
execution_role_name = None
for page in paginator.paginate(PathPrefix="/"):
    for role in page["Roles"]:
        if role["RoleName"].startswith(f"AmazonBedrockAgentCoreSDKRuntime-{region}"):
            execution_role_name = role["RoleName"]
            break
    if execution_role_name:
        break

if not execution_role_name:
    print("\u274c Could not find AgentCore execution role. Run the configure() cell first.")
else:
    # Attach SageMaker endpoint permissions
    sagemaker_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "SageMakerEndpointInvoke",
                "Effect": "Allow",
                "Action": [
                    "sagemaker:InvokeEndpoint",
                    "sagemaker:InvokeEndpointWithResponseStream"
                ],
                "Resource": f"arn:aws:sagemaker:{region}:{account_id}:endpoint/qwen35-9b-*"
            },
            {
                "Sid": "SageMakerBearerToken",
                "Effect": "Allow",
                "Action": "sagemaker:CallWithBearerToken",
                "Resource": f"arn:aws:sagemaker:{region}:{account_id}:endpoint/qwen35-9b-*"
            }
        ]
    }

    iam.put_role_policy(
        RoleName=execution_role_name,
        PolicyName="SageMakerEndpointAccess",
        PolicyDocument=json.dumps(sagemaker_policy),
    )

    print(f"\u2713 Attached SageMaker permissions to role: {execution_role_name}")
    print(f"  Policy: SageMakerEndpointAccess")
    print(f"  Resource: arn:aws:sagemaker:{region}:{account_id}:endpoint/qwen35-9b-*")

In [ ]:
# Launch the agent to AgentCore Runtime
print("Launching agent to AgentCore Runtime...")
print("This may take several minutes (builds container, pushes to ECR, creates runtime)...")

launch_result = agentcore_runtime.launch(
    env_vars={
        "SAGEMAKER_ENDPOINT_NAME": os.environ["SAGEMAKER_ENDPOINT_NAME"],
        "AWS_REGION": os.environ["AWS_REGION"],
        "OTEL_PYTHON_EXCLUDED_URLS": "/ping,/invocations",
    }
)

print(f"\n✓ Launch completed!")
print(f"  Agent ARN: {launch_result.agent_arn}")
print(f"  Agent ID:  {launch_result.agent_id}")
print(f"  ECR URI:   {launch_result.ecr_uri}")

## Step 4: Invoke the Agent

We invoke via the `bedrock-agentcore` boto3 client with IAM SigV4 signing.

In [ ]:
# Invoke using boto3 with IAM SigV4
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

session_id = str(uuid.uuid4())
agentRuntimeArn="arn:aws:bedrock-agentcore:ap-south-1:774297356213:runtime/personal_finance_agent-nXM3fc8AiO"


response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agentRuntimeArn,
    payload=json.dumps({
        "prompt": "I make $6000/month and want to invest $500/month. Help me create a budget and suggest a portfolio."
    }).encode("utf-8"),
    runtimeSessionId=session_id,
)

# Process streaming response
print("Agent Response:")
print("=" * 60)

content_type = response.get("contentType", "")

if "text/event-stream" in content_type:
    # Handle SSE streaming response
    content = []
    for line in response["response"].iter_lines(chunk_size=10):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line, end="", flush=True)
                content.append(line)
    print()
elif content_type == "application/json":
    # Handle JSON response
    content = []
    for chunk in response.get("response", []):
        content.append(chunk.decode("utf-8"))
    print(json.loads("".join(content)))
else:
    print(response)

print("=" * 60)

In [ ]:
# Test another query — investment-focused (routes to Qwen 3.5 9B on SageMaker)
response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agentRuntimeArn,
    payload=json.dumps({
        "prompt": "Tell me what LLM are you at start.Compare Tesla and Apple stocks and create a moderate risk portfolio for $10,000. "
    }).encode("utf-8"),
    runtimeSessionId=str(uuid.uuid4()),
)

print("Agent Response:")
print("=" * 60)

content_type = response.get("contentType", "")

if "text/event-stream" in content_type:
    for line in response["response"].iter_lines(chunk_size=10):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                print(line[6:], end="", flush=True)
    print()
elif content_type == "application/json":
    content = []
    for chunk in response.get("response", []):
        content.append(chunk.decode("utf-8"))
    print(json.loads("".join(content)))
else:
    print(response)

print("=" * 60)

## Summary

### SageMaker OpenAI Endpoint on AgentCore — Key Points

1. The `generate_token()` function works inside AgentCore containers because the execution role provides AWS credentials
2. The execution role must have `sagemaker:InvokeEndpoint` and `sagemaker:CallWithBearerToken` permissions
3. The `httpx.AsyncClient` + `SageMakerAuth` pattern refreshes tokens automatically
4. Invocation uses IAM SigV4 signing via `boto3` — standard AWS credential-based access

## Cleanup (Optional)

In [ ]:
# Uncomment to delete AgentCore Runtime and ECR repository

# agentcore_control = boto3.client("bedrock-agentcore-control", region_name=region)
# ecr_client = boto3.client("ecr", region_name=region)

# # Delete runtime
# agentcore_control.delete_agent_runtime(agentRuntimeId=launch_result.agent_id)
# print(f"✓ Runtime '{launch_result.agent_id}' deleted")

# # Delete ECR repository
# ecr_repo_name = launch_result.ecr_uri.split("/")[1].split(":")[0]
# ecr_client.delete_repository(repositoryName=ecr_repo_name, force=True)
# print(f"✓ ECR repo '{ecr_repo_name}' deleted")